In [ ]:
import time
# autoreload
%load_ext autoreload
%autoreload 2

#from scipy import signal
#from scipy import interpolate
#from scipy import ndimage
import numpy as np
#import pycatch22 
#from sktime.transformations.panel import catch22
#import tsfresh
from tqdm import tqdm
import sys, os
import pandas as pd 
import dotenv
import random
load_dotenv = dotenv.load_dotenv('../.env')

# load local library
from timex import clustering
from timex import preprocessing

import gc
import seaborn as sns
import matplotlib.pyplot as plt

from collections import defaultdict

import datetime
from time import sleep

from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
import json

# limit to 8 threads
os.environ["OMP_NUM_THREADS"] = "8"
os.environ["OPENBLAS_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"
os.environ["VECLIB_MAXIMUM_THREADS"] = "8"
os.environ["NUMEXPR_NUM_THREADS"] = "8"

In [ ]:
AKI_PATH = os.environ['AKI_PATH_NEW']
os.chdir(AKI_PATH)

In [ ]:
AKI_PATH


In [ ]:
os.listdir('.')

In [ ]:
ts_data = pd.read_parquet(os.path.join(AKI_PATH, 'cleaned_DV_LCMM_data.parquet'))
ts_data = ts_data.rename(columns={"Time_since_index_FU_days": "Time_days"})
ts_data.ID = ts_data.ID.astype('int64')
ts_data['dataset_nr'] = ts_data['dataset_nr'].fillna(-1).astype('int64')


In [ ]:
ts_data.groupby('dataset_nr').ID.nunique(), ts_data.ID.nunique()

In [ ]:
MIN_TIME = 365 * 1 # days
MAX_TIME = 365 * 10 # days
MIN_MEAS_COUNT = 3 # measurements
INTERP_RES = 1
SMOOTHING_WINDOW = 365 # in days: 4 * INTERP_RES = 360
SMOOTHING_TYPE = 'gaussian_kernel'  # 'gaussian_kernel' or 'rolling_mean'
META_KEYS = ['ID', 'Time_days']
RAW_VAL_COL = 'eGFRcr_CKDEpi2009'
INT_VAL_COL = 'eGFR_int'
SM30_VAL_COL = 'eGFR_SW30'
SM365_VAL_COL = 'eGFR_SW365'

DS_SELECTION =  [list(range(1,K+1)) for K in range(1,9)]  # which datasets to include in the analysis
SELECTION_RES = 1 # 180, 90, 60, 30
CLUSTER_NUMS =  [2, 4, 6, 8, 10, 12, 14, 16]

CLUSTERING_ALGO='gmm'
CLUSTER_KWARGS={"reg_covar": 1e-5, "covariance_type": "diag"}

ADD_TS_META = True
EXTRACTORS = ['custom', 'catch22', 'tsfel'] # tsfel



In [ ]:
ts_data_df = ts_data[['ID', 'Time_days', 'eGFRcr_CKDEpi2009', 'dataset_nr']].dropna(subset=['eGFRcr_CKDEpi2009'])

In [ ]:
meta_str = "_wMeta" if ADD_TS_META else "_noMeta"
file_dir = f"Results/{"_".join(EXTRACTORS)}{meta_str}_TR{SELECTION_RES}"
if not os.path.exists(file_dir):
    os.makedirs(file_dir)

In [ ]:
for NUM_CLUSTERS in CLUSTER_NUMS:
    for DS_SEL in DS_SELECTION:
        print(f'Running clustering for DS{DS_SEL}_C{NUM_CLUSTERS}_TR{SELECTION_RES}')


        Sel_IDS = ts_data[ts_data['dataset_nr'].isin(DS_SEL)].ID.unique()
        ts_data_run = ts_data_df[ts_data_df.ID.isin(Sel_IDS)]

        SETTINGS_DICT = {
            'MIN_TIME': MIN_TIME,
            'MAX_TIME': MAX_TIME,
            'MIN_MEAS_COUNT': MIN_MEAS_COUNT,
            'INTERP_RES': INTERP_RES,
            'SMOOTHING_WINDOW': SMOOTHING_WINDOW,
            'SMOOTHING_TYPE': SMOOTHING_TYPE,
            'META_KEYS': META_KEYS,
            'RAW_VAL_COL': RAW_VAL_COL,
            'INT_VAL_COL': INT_VAL_COL,
            'SM30_VAL_COL': SM30_VAL_COL,
            'SM365_VAL_COL': SM365_VAL_COL,
            'DS_SELECTION': DS_SEL,
            'SELECTION_RES': SELECTION_RES,
            'NUM_CLUSTERS': NUM_CLUSTERS,
            'CLUSTERING_ALGO': CLUSTERING_ALGO,
            'CLUSTER_KWARGS': CLUSTER_KWARGS,
            'ADD_TS_META': ADD_TS_META,
            'EXTRACTORS': EXTRACTORS
        }

        ts_clusterer = clustering.CrossSectionalClustering(smoothing=True, 
                                                        smoothing_type=SMOOTHING_TYPE,
                                                        smoothing_window_size=SMOOTHING_WINDOW,
                                                        n_skip=3,
                                                        interpolation=True, 
                                                        interpolation_resolution=INTERP_RES,
                                                        interpolation_keep_init=True,
                                                        analysis_resolution=SELECTION_RES,
                                                        min_measurements_per_id=MIN_MEAS_COUNT, 
                                                        min_time=MIN_TIME,
                                                        max_time=MAX_TIME,
                                                        clustering_algorithm=CLUSTERING_ALGO,
                                                        n_clusters=NUM_CLUSTERS, 
                                                        cluster_kwargs=CLUSTER_KWARGS,
                                                        id_column='ID', 
                                                        time_column='Time_days',
                                                        feature_columns=[RAW_VAL_COL],
                                                        imputation_method='knn',
                                                        cross_standardisation=True,
                                                        normalise_timeseries= "group",
                                                        normalisation_method="standard",
                                                        add_ts_meta=ADD_TS_META,
                                                        extractors=EXTRACTORS,
                                                        verbose=True)


        ts_clusterer.fit(ts_data_run)

        SELECTION_RES_STR = str(SELECTION_RES) if SELECTION_RES>1 else '0'
        ts_label_df = pd.read_parquet(f'analysed_DV_LCMM_outcome_TR{SELECTION_RES_STR}d.parquet')
        ts_label_df['ID'] = ts_label_df.ID.astype(int)
        ts_label_df.set_index('ID', inplace=True)
        ts_label_df.dropna(how='all', inplace=True)

       
        class_string = f'ds{''.join([str(c) for c in DS_SEL])}_M2splines_Llin_eGFR_C{NUM_CLUSTERS}_TR{SELECTION_RES}_Class' 
        proba_string = f'ds{''.join([str(c) for c in DS_SEL])}_M2splines_Llin_eGFR_C{NUM_CLUSTERS}_TR{SELECTION_RES}_max_prob'    
        res_cluster = pd.DataFrame(zip(ts_clusterer.ts_cross_combined.index, ts_clusterer.predict()), columns=['ID', 'cluster'])
        res_cluster['cluster'] = res_cluster['cluster'].astype(int)

        res_cluster_proba = pd.DataFrame()
        res_cluster_proba['ID'] = ts_clusterer.ts_cross_combined.index
        res_cluster_proba[[f'cluster_prob_{i}' for i in range(NUM_CLUSTERS)]] =  ts_clusterer.predict_proba()

        try:
            ts_label_df_LCMM = ts_label_df.dropna(subset=[class_string])[[proba_string, class_string]]
            ts_label_df_LCMM[class_string] = ts_label_df_LCMM[class_string].astype('int')
            
            ts_label_df_LCMM = ts_label_df_LCMM.rename(columns={
                proba_string: 'LCMM_max_prob',
                class_string: 'LCMM_Class'
                })


            res_final = ts_data_run.merge(res_cluster, how='left', left_on='ID', right_on='ID').dropna(subset='ID')
            res_final = res_final.merge(ts_label_df_LCMM, how='left', left_on='ID', right_index=True).dropna(subset=['cluster'])\
                                    .astype({'cluster': 'int'})

            res_final_ = res_final.groupby('ID')[['LCMM_Class', 'cluster']].first()
            res_final_['LCMM_Class'] = res_final_['LCMM_Class'].astype(int) - 1

            external_scores = {
                'ari': adjusted_rand_score(res_final_['cluster'], res_final_['LCMM_Class']), 
                'ami': adjusted_mutual_info_score(res_final_['cluster'], res_final_['LCMM_Class'])
            }
        except Exception as e:
            print(f'Could not compute external scores: {e}')
            external_scores = {
                'ari': None,
                'ami': None
            }


        internal_scores = ts_clusterer.get_scores()

        # combine all scores and timings in a single dictionary
        scores_dict = {
            'external_scores': external_scores,
            'internal_scores': internal_scores,
            'timings': ts_clusterer.timings,
            'settings': SETTINGS_DICT
        }

        # write scores to a jsonl file, appending if the file already exists
        with open(f'Results/{"_".join(EXTRACTORS)}clustering_scores_log.jsonl', 'a') as f:
            f.write(json.dumps(scores_dict) + '\n') 


        res_cluster_proba.to_csv(f'{file_dir}/cluster_probs_ds{''.join([str(c) for c in DS_SEL])}_C{NUM_CLUSTERS}_TR{SELECTION_RES}.csv', sep=';')

In [ ]:
# # plot 10 random samples
# for s in ts_clusterer.ts_filtered.sample(n=1)['ID']:
#     tsv = ts_clusterer.ts_filtered.query(f'ID=={s}')['eGFRcr_CKDEpi2009']
#     tst = ts_clusterer.ts_filtered.query(f'ID=={s}')['Time_days']
#     plt.plot(tst, tsv, color='red', alpha=0.7)

#     tsv = ts_clusterer.ts_smoothed.query(f'ID=={s}')['eGFRcr_CKDEpi2009']
#     tst = ts_clusterer.ts_smoothed.query(f'ID=={s}')['Time_days']
#     plt.plot(tst, tsv, color='green', alpha=0.7)

#     tsv = ts_clusterer.ts_smoothed_filtered.query(f'ID=={s}')['eGFRcr_CKDEpi2009']
#     tst = ts_clusterer.ts_smoothed_filtered.query(f'ID=={s}')['Time_days']
#     plt.plot(tst, tsv, color='blue', alpha=0.7)